# DES Hub Netbacks - Marginal Terminals

Determining which terminals' Moneyness metric have been most marginal historically. 

Moneyness metric = Front Month hub price - variable regas costs - SparkNWE/SWE DES LNG price

For a full explanation of how to import our DES Hub Netbacks or SparkNWE data, please refer to our Python Jupyter Notebook Code Samples:

https://www.sparkcommodities.com/api/code-examples/jupyter.html

### Have any questions?

If you have any questions regarding our API, or need help accessing specific datasets, please contact us at:

__data@sparkcommodities.com__


## 1. Importing Data

Here we define the functions that allow us to retrieve the valid credentials to access the Spark API.

This section can remain unchanged for most Spark API users.

In [ ]:
import json
import os
import sys
import pandas as pd
import numpy as np
from base64 import b64encode
from urllib.parse import urljoin
from io import StringIO
import datetime

try:
    from urllib import request, parse
    from urllib.error import HTTPError
except ImportError:
    raise RuntimeError("Python 3 required")

In [ ]:
API_BASE_URL = "https://api.sparkcommodities.com"


def retrieve_credentials(file_path=None):
    """
    Find credentials either by reading the client_credentials file or reading
    environment variables
    """
    if file_path is None:
        client_id = os.getenv("SPARK_CLIENT_ID")
        client_secret = os.getenv("SPARK_CLIENT_SECRET")
        if not client_id or not client_secret:
            raise RuntimeError(
                "SPARK_CLIENT_ID and SPARK_CLIENT_SECRET environment vars required"
            )
    else:
        # Parse the file
        if not os.path.isfile(file_path):
            raise RuntimeError("The file {} doesn't exist".format(file_path))

        with open(file_path) as fp:
            lines = [l.replace("\n", "") for l in fp.readlines()]

        if lines[0] in ("clientId,clientSecret", "client_id,client_secret"):
            client_id, client_secret = lines[1].split(",")
        else:
            print("First line read: '{}'".format(lines[0]))
            raise RuntimeError(
                "The specified file {} doesn't look like to be a Spark API client "
                "credentials file".format(file_path)
            )

    print(">>>> Found credentials!")
    
    return client_id, client_secret


def do_api_post_query(uri, body, headers):
    """
    OAuth2 authentication requires a POST request with client credentials before accessing the API.
    This POST request will return an Access Token which will be used for the API GET request.
    """
    url = urljoin(API_BASE_URL, uri)

    data = json.dumps(body).encode("utf-8")

    # HTTP POST request
    req = request.Request(url, data=data, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 201. Raise an error if this is not the case
    assert response.status == 201, resp_content

    # The server returned a JSON response
    content = json.loads(resp_content)

    return content


def do_api_get_query(uri, access_token, format='json'):
    """
    After receiving an Access Token, we can request information from the API.
    Supports both JSON (default) and CSV responses.
    """
    url = urljoin(API_BASE_URL, uri)

    if format == 'json':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "application/json",
        }
    elif format == 'csv':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "text/csv",
        }
    else:
        raise ValueError("The format parameter only takes 'csv' or 'json' as inputs")

    print("Fetching {}".format(url))

    # HTTP GET request
    req = request.Request(url, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 200. Raise an error if this is not the case
    assert response.status == 200, resp_content

    # Storing response based on requested format
    if format == 'json':
        content = json.loads(resp_content)
    elif format == 'csv':
        content = resp_content

    return content


def get_access_token(client_id, client_secret):

    payload = "{}:{}".format(client_id, client_secret).encode()
    headers = {
        "Authorization": b64encode(payload).decode(),
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    body = {
        "grantType": "clientCredentials",
    }

    content = do_api_post_query(uri="/oauth/token/", body=body, headers=headers)

    print(
        ">>>> Successfully fetched an access token {}****, valid {} seconds.".format(
            content["accessToken"][:5], content["expiresIn"]
        )
    )

    return content["accessToken"]

## N.B. Credentials

Here we call the above functions, and input the file path to our credentials.

N.B. You must have downloaded your client credentials CSV file before proceeding. Please refer to the API documentation if you have not dowloaded them already.

The code then prints the available prices that are callable from the API, and their corresponding Python ticker names are displayed as a list at the bottom of the Output.

In [ ]:
# Insert file path to your client credentials here
client_id, client_secret = retrieve_credentials(file_path="/tmp/client_credentials.csv")

# Authenticate:
access_token = get_access_token(client_id, client_secret)

## 2. DES Hub Netbacks

Calling the DES Hub Netbacks data, and sorting into a Historical DataFrame

In [ ]:
## Defining the function

def fetch_historical_price_releases(access_token, unit, latest=False, start=None, end=None, limit=None, offset=None, terminal_uuid=None, format='json'):
    
    # if latest=True, set query_params to call the ".../latest/" endpoint
    if latest == True:
        query_params = 'latest/'
        query_params += "?unit={}".format(unit)
    
    else:
        query_params = "?unit={}".format(unit)

    if start is not None:
        query_params += "&start={}".format(start)
    if end is not None:
        query_params += "&end={}".format(end)
    
    if terminal_uuid is not None:
        query_params += "&terminal-uuid={}".format(terminal_uuid)
    

    uri = "/v1.0/lng/access/des-hub-netbacks/{}".format(query_params)

    content = do_api_get_query(
        uri=uri, access_token=access_token, format=format
    )

    if format == 'json':
        data = content
    elif format == 'csv':
        # if there's no data to show, returns raw response (empty string) and "No Data to Show" message
        if len(content) == 0:
            data = content
            print('No Data to Show')
        else:
            data = content.decode('utf-8')
            data = pd.read_csv(StringIO(data))  # automatically converting into a Pandas DataFrame when choosing CSV format

    return data


### Historical Data Function

Currently, a maximum of 1 year's worth of historical data can be called at one time due to the size of the data file. 

If more data points are required, the below code can be used: the function calls the historical data 1 year at a time and combines the data into one Pandas DataFrame

In [ ]:
import datetime

def loop_historical_data(access_token, hist_unit, hist_start, hist_end):
    
    hist_diff = (datetime.datetime.strptime(hist_end, '%Y-%m-%d') - datetime.datetime.strptime(hist_start, '%Y-%m-%d')).days
    t = 0

    starts = []
    ends = []

    w = 365

    while t < hist_diff:
        if t == 0:
            diff_end = datetime.datetime.strftime(datetime.datetime.strptime(hist_start, '%Y-%m-%d') + pd.Timedelta(days=w), '%Y-%m-%d')
            hist_df = fetch_historical_price_releases(access_token, unit=hist_unit, start=hist_start, end=diff_end, format='csv')

            starts.append(hist_start)
            ends.append(diff_end)

        else:
            if t < hist_diff-w:
                diff_start = datetime.datetime.strftime(datetime.datetime.strptime(hist_start, '%Y-%m-%d') + pd.Timedelta(days=t+1), '%Y-%m-%d')
                diff_end = datetime.datetime.strftime(datetime.datetime.strptime(diff_start, '%Y-%m-%d') + pd.Timedelta(days=w), '%Y-%m-%d')
                historical_addition = fetch_historical_price_releases(access_token, unit=hist_unit, start=diff_start, end=diff_end, format='csv')
                try:
                    hist_df = pd.concat([hist_df,historical_addition])
                #exception if hist_df is empty
                except:
                    hist_df = historical_addition.copy()
                starts.append(hist_start)
                ends.append(diff_end)
            else:
                diff_start = datetime.datetime.strftime(datetime.datetime.strptime(hist_start, '%Y-%m-%d') + pd.Timedelta(days=t+1), '%Y-%m-%d')
                diff_end = datetime.datetime.strftime(datetime.datetime.strptime(diff_start, '%Y-%m-%d') + pd.Timedelta(days=hist_diff-t), '%Y-%m-%d')
                historical_addition = fetch_historical_price_releases(access_token, unit=hist_unit, start=diff_start, end=diff_end, format='csv')
                hist_df = pd.concat([hist_df, historical_addition])
                starts.append(hist_start)
                ends.append(diff_end)
        
        #looping by year
        t += w

    for c in list(hist_df.columns)[8:]:
        hist_df[c] = pd.to_numeric(hist_df[c])
    hist_df['ReleaseDate'] = pd.to_datetime(hist_df['ReleaseDate'])

    return hist_df

In [ ]:
deshub_df = loop_historical_data(access_token, hist_unit='usd-per-mmbtu', hist_start='2024-01-01', hist_end='2026-01-10')
deshub_df.head(5)

# 3. SparkNWE & SparkSWE - Data Import

Calling the SparkNWE & SparkSWE front month prices (in TTF basis format) to compare against the terminal DES Hub netbacks and determine which terminals are "in the money"

In [ ]:
# Calling contracts endpoint to import cargo data
def fetch_cargo_releases(access_token, ticker, limit=4, offset=None):

    print(">>>> Get price releases for {}".format(ticker))

    query_params = "?limit={}".format(limit)
    if offset is not None:
        query_params += "&offset={}".format(offset)

    content = do_api_get_query(
        uri="/v1.0/contracts/{}/price-releases/{}".format(ticker, query_params),
        access_token=access_token,
    )

    my_dict = content['data']
    
    return my_dict

# Function to import data and then sort into a DataFrame
def cargo_to_dataframe(access_token, ticker, limit, month):

    # imports front month or forward curve prices, depending on the "month" user input
    if month == 'M+1':
        full_tick = ticker + '-b-f'
        hist_data = fetch_cargo_releases(access_token, full_tick, limit)
    else:
        full_tick = ticker + '-b-fo'
        hist_data = fetch_cargo_releases(access_token, full_tick, limit)
         

    release_dates = []
    period_start = []
    ticker = []
    spark = []

    spark_min = []
    spark_max = []
    cal_month = []

    # iterating through historical data points to fetch relevant data
    for release in hist_data:
            release_date = release["releaseDate"]
            ticker.append(release['contractId'])
            release_dates.append(release_date)

            mi = int(month[-1])-2

            data_point = release['data'][0]['dataPoints'][mi]

            period_start_at = data_point["deliveryPeriod"]["startAt"]
            period_start.append(period_start_at)
            
            spark.append(data_point['derivedPrices']['usdPerMMBtu']['spark'])
            spark_min.append(data_point['derivedPrices']['usdPerMMBtu']['sparkMin'])
            spark_max.append(data_point['derivedPrices']['usdPerMMBtu']['sparkMax'])
            
            cal_month.append(datetime.datetime.strptime(period_start_at, '%Y-%m-%d').strftime('%b-%Y'))
                

    # Converting into DataFrame
    hist_df = pd.DataFrame({
        'ReleaseDate': release_dates,
        'Ticker': ticker,
        'PeriodStart': period_start,
        'Price': spark,
        })
        
    
    hist_df['Price'] = pd.to_numeric(hist_df['Price'])
    hist_df['ReleaseDate'] = pd.to_datetime(hist_df['ReleaseDate'])

    hist_df['ReleaseDate'] = hist_df['ReleaseDate'].dt.tz_localize(None) 

    return hist_df

# 4. Analysis

- Input contract month required
- Subtracting the SparkNWE/SWE prices from the DES Hub Netbacks to determine whether terminals are "in" or "out of the money".
    - This will be referred to as the Moneyness metric
- Determining which terminal is considered the most "marginal" - i.e. closest to Moneyness=0
- Plotting which terminals have been the most marginal historically


### Inputs

In [ ]:
# Choose which forward month you'd like to analyse - either front month ("M+1") or any other month up until M+11
month = 'M+1'

# Here we define which terminals we want to use in the analytics. The default is all terminals, but you can choose a subset if preferred (as demonstrated in comment below)
deshub_terms = list(deshub_df['TerminalName'].unique())
terms = deshub_terms.copy()
#terms = ['gate', 'dunkerque', 'zeebrugge']

### Data Calling & Analytical Procedures

In [ ]:
# Import NWE/SWE LNG prices
sparknwe = cargo_to_dataframe(access_token, 'sparknwe', 2000, month=month)
sparkswe = cargo_to_dataframe(access_token, 'sparkswe', 2000, month=month)

# retrieve the same amount of historical data for SparkSWE as SparkNWE 
sparkswe = sparkswe[sparkswe['ReleaseDate'] >= sparknwe['ReleaseDate'].iloc[-1]].copy()

# Combine datasets and backfill SWE data as needed (due to reduced assessment frequency)
cargo_df = pd.merge(sparknwe, sparkswe, how='left', on='ReleaseDate')
cargo_df['Price_y'] = cargo_df['Price_y'].bfill().copy()

cargo_df = cargo_df[['ReleaseDate', 'Price_x', 'Price_y']].copy()
cargo_df = cargo_df.rename(columns={'Price_x': 'SparkNWE',
                                    'Price_y': 'SparkSWE'})

In [ ]:
# Defining which terminals belong to NWE or SWE, so that the relevant DES LNG price can be subtracted to calculate the terminals' Moneyness
terminal_region_dict = {
    'Gate': 'nwe',
    'Isle of Grain': 'nwe',
    'Zeebrugge': 'nwe',
    'South Hook': 'nwe',
    'Dunkerque': 'nwe',
    'Le Havre': 'nwe',
    'Montoir': 'nwe',
    'EemsEnergyTerminal': 'nwe',
    'Brunsbuttel': 'nwe',
    'Deutsche Ostsee': 'nwe',
    'Deutsche Ostsee Phase 1': 'nwe',
    'Wilhelmshaven 1': 'nwe',
    'Wilhelmshaven 2': 'nwe',
    'Stade':'nwe',
    'Fos Cavaou': 'swe',
    'Adriatic': 'swe',
    'OLT Toscana': 'swe',
    'Piombino': 'swe',
    'Ravenna': 'swe',
    'Spain TVB': 'swe'
}


In [ ]:
# Setting which costs to consider sunk and using a boolean variable to control whether these costs are removed from the DES Hub Netback values

sunk_bool = True

sunk_costs_list = ['SlotUnloadStorageRegas','SlotBerth', 'SlotBerthUnloadStorageRegas', 'SlotLtCapacityEstimate']

In [ ]:
# Initialising the "month" dataframe, which uses the "month" user input to create a DataFrame with all the relevant DES Hub netbacks data for that month for each terminal
# the Gate DES Hub Netbacks data is used to set the "ReleaseDate" and "DeliveryMonthIndex" columns as Gate has the longest historical dataset
month_df = deshub_df[(deshub_df['TerminalName'] == 'Gate') & (deshub_df['DeliveryMonthIndex'] == month)][['ReleaseDate', 'DeliveryMonthIndex', 'NetbackTTFBasis']]

# Removing sunk costs for Gate only
if sunk_bool == True:
    month_df['NetbackTTFBasis'] = month_df['NetbackTTFBasis'] +  \
        deshub_df[(deshub_df['TerminalName'] == 'Gate') & (deshub_df['DeliveryMonthIndex'] == month)][sunk_costs_list].sum(axis=1)

month_df = month_df.rename(columns={'NetbackTTFBasis':'Gate'})

# defining a new list of terminals without "Gate" in it
terms2 = [x if x != 'Gate' else None for x in deshub_terms]

# iterating through list of terminals and adding data to the Terminal Moneyness dataframe
for t in terms2:
    if t is not None:
        tdf = deshub_df[(deshub_df['TerminalName'] == t) & (deshub_df['DeliveryMonthIndex'] == month)][['ReleaseDate', 'NetbackTTFBasis']]
        if sunk_bool == True:
            tdf['NetbackTTFBasis'] = tdf['NetbackTTFBasis'] +  \
            deshub_df[(deshub_df['TerminalName'] == t) & (deshub_df['DeliveryMonthIndex'] == month)][sunk_costs_list].sum(axis=1)
        month_df = month_df.merge(tdf, on='ReleaseDate', how='left')
        month_df = month_df.rename(columns={'NetbackTTFBasis':t})

# Calculating the Average, Minimum and Maximum Moneyness values for all terminals (i.e. for Europe)
month_df['Ave'] = month_df[deshub_terms].mean(axis=1)
month_df['Min'] = month_df[deshub_terms].min(axis=1)
month_df['Max'] = month_df[deshub_terms].max(axis=1)

# Merging Cargo prices with the DataFrame, and backfilling data as needed so that the datasets are the same length (needed due to differing price release frequency)
month_df = pd.merge(month_df,cargo_df, how='left', left_on='ReleaseDate', right_on='ReleaseDate')
month_df['SparkNWE'] = month_df['SparkNWE'].bfill().copy()
month_df['SparkSWE'] = month_df['SparkSWE'].bfill().copy()

month_df = month_df.sort_values('ReleaseDate', ascending=False)

In [ ]:
# Creating a Moneyness dataframe, subtracting NWE/SWE prices from each terminals' DES Hub netbacks data.
# The use of NWE or SWE prices for each terminal is set by the "terminal_region_dict" defined earlier in the script 
money_df = month_df[['ReleaseDate', 'DeliveryMonthIndex', 'SparkNWE', 'SparkSWE']].copy()

for t in terms:
    if terminal_region_dict[t] == 'nwe':
        money_df[t] = month_df[t].copy() - month_df['SparkNWE'].copy()
    elif terminal_region_dict[t] == 'swe':
        money_df[t] = month_df[t].copy() - month_df['SparkSWE'].copy()
    else:
        money_df[t] = month_df[t].copy() - month_df['SparkNWE'].copy()

In [ ]:
# Calculating the Average, Min and Max Moneyness metric over all terminals
money_df['Ave'] = money_df[terms].mean(axis=1)
money_df['Min'] = money_df[terms].min(axis=1)
money_df['Max'] = money_df[terms].max(axis=1)
    
# Determining which is the marginal terminal for each historical date
money_df['Marginal Terminal'] = money_df[terms].abs().idxmin(axis="columns").copy()

# saving list of marginal terminals
marg_list = list(money_df['Marginal Terminal'].unique())

In [ ]:
money_df.head(5)

### Plotting

Plotting the average European WTP and min/max range, as well as the marginal terminal going back historically

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mtick

sns.set_style('whitegrid')

fig, ax = plt.subplots(figsize=(15, 7))

ax.plot(money_df['ReleaseDate'], money_df['Ave'], color='mediumseagreen', linewidth=2.0, label='European Average WTP')

# Option to plot min/max range of European Average WTP
#ax.plot(money_df['Release Date'], money_df['Min'], color='steelblue', linewidth=1.0, alpha=0.06)
#ax.plot(money_df['Release Date'], money_df['Max'], color='steelblue', linewidth=1.0, alpha=0.06)
#ax.fill_between(money_df['Release Date'], money_df['Min'], money_df['Max'], color='steelblue', alpha=0.2)

for m in marg_list:
    tdf = money_df[money_df['Marginal Terminal'] == m][['ReleaseDate', m]]
    ax.scatter(tdf['ReleaseDate'], tdf[m], label=m)

# Shading <0 area of the chart
negrange = [money_df['ReleaseDate'].iloc[-1] -pd.Timedelta(days=20), money_df['ReleaseDate'].iloc[0] +pd.Timedelta(days=20)]
ax.plot(negrange,[-1.0,-1.0], color='red', alpha=0.05)
ax.plot(negrange,[0,0], color='red', alpha=0.05)
ax.fill_between(negrange, 0, -1.0, color='red', alpha=0.05)

# Chart Aesthetics
plt.title('European Average Moneyness vs Marginal Cargo')
plt.legend(loc='upper left')
plt.ylim(-0.4,0.6)

sns.despine(left=True, bottom=True)